# 합성어 경음화 환경 검색

**작성일**: 2026-02-10  
**목적**: 합성어 경음화 환경 검색 및 형태론적 조건 분석  
**관련**: 34_n_insertion_v2.ipynb, 35_nl_ln_nasalization.ipynb

---

## 🎯 검색 대상

### 1. 음운론적 조건 (형태소 경계)

| morph1 끝 | morph2 시작 | 예시 | 경음화 |
|-----------|-------------|------|--------|
| 모음 (a/e/i/o/u) | 평음 (k/t/p/j/s) | 바-다가 | [바따가] |
| 비음 (m/n/ng) | 평음 (k/t/p/j/s) | 손-가락 | [손까락] |
| 유음 (l/r) | 평음 (k/t/p/j/s) | 물-질 | [물찔] |
| 저해음 (k/t/p/s 등) | 평음 (k/t/p/j/s) | 학-교 | [학꾜] |

### 2. 형태론적 조건 (어종 조합)

| 유형 | morph1 | morph2 | 예시 |
|------|--------|--------|------|
| **Type A** | 고유어 | 고유어 | 손-가락, 밤-길 |
| **Type B** | 한자어/외래어 | 고유어 | 학-교, 아파트-단지 |
| **Type C** | any | 한자어 | 물-질, 공-간 |

---

## 📊 출력 정보

### 환경 정보
- morph1_final_type: vowel / nasal / liquid / obstruent
- morph2_initial: k / t / p / j / s (평음만)

### 경음화 정보
- detected_fortis: yes / no / unknown
- fortis_type: k→GG, t→DD 등

### 형태론 정보
- morph1_origin, morph2_origin: 고유어/한자어/외래어
- compound_type: A / B / C

### 빈도 정보
- LS, MP, Freq_2009

---

## 1️⃣ 환경 설정

In [1]:
# 1.1 Google Drive 마운트
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except:
    IN_COLAB = False
    print("로컬 환경")

Mounted at /content/drive


In [2]:
# 1.2 경로 설정
if IN_COLAB:
    PROJECT_ROOT = '/content/drive/MyDrive/DATA_2026'
else:
    PROJECT_ROOT = 'g:/내 드라이브/DATA_2026'

V7_LEXICON = f'{PROJECT_ROOT}/10_dictionary_build/output/04_v7_lexicon.csv'
RESULT_DIR = f'{PROJECT_ROOT}/30_search_dictionary/search_results'

print(f"v7: {V7_LEXICON}")
print(f"결과: {RESULT_DIR}")

v7: /content/drive/MyDrive/DATA_2026/10_dictionary_build/output/04_v7_lexicon.csv
결과: /content/drive/MyDrive/DATA_2026/30_search_dictionary/search_results


In [3]:
# 1.3 라이브러리
import pandas as pd
import numpy as np
from datetime import datetime
from pathlib import Path
import re
import os

Path(RESULT_DIR).mkdir(parents=True, exist_ok=True)

# 공유 유틸리티 모듈 로드
os.chdir(f'{PROJECT_ROOT}/30_search_dictionary')
%run utils_phonology.py
print("준비 완료 (utils_phonology.py 로드됨)")

[OK] utils_phonology.py 로드 완료
   함수 34개
준비 완료 (utils_phonology.py 로드됨)


---

## 2️⃣ 데이터 로드

In [4]:
# 2.1 v7 Lexicon 로드
df_v7 = pd.read_csv(V7_LEXICON, encoding='utf-8-sig', low_memory=False)
print(f"v7 Lexicon: {len(df_v7):,}개")
print(f"dict_morph 있는 행: {df_v7['dict_morph'].notna().sum():,}개")

v7 Lexicon: 528,088개
dict_morph 있는 행: 323,146개


# 2.2 형태소 어종 딕셔너리 구축 ⭐

**중요**: 각 형태소의 어종을 정확하게 파악하기 위해 v7에서 단일 형태소 추출

In [5]:
# 2.2.1 어종 딕셔너리 구축 (sense_no 기반)
sense_origin_dict = build_sense_origin_dict(df_v7)

sense_no 어종 딕셔너리: 74개


In [6]:
# 2.2.2 Origin 필드 진단
print("=" * 70)
print("Origin 필드 진단")
print("=" * 70)

# 전체 명사에서 origin 필드 통계
df_noun_all = df_v7[df_v7['pos'] == '명사'].copy()
print(f"\n전체 명사: {len(df_noun_all):,}개")
print(f"origin 필드 non-null: {df_noun_all['origin'].notna().sum():,}개")
print(f"origin 필드 null/empty: {df_noun_all['origin'].isna().sum():,}개")

# origin 값 분포 (상위 20개)
print(f"\norigin 필드 값 분포 (상위 20개):")
print(df_noun_all['origin'].value_counts().head(20))

# origin_lang 필드도 확인
print(f"\norigin_lang 필드 값 분포 (상위 20개):")
print(df_noun_all['origin_lang'].value_counts().head(20))

# 샘플 단어들의 origin 정보 확인
print(f"\n샘플 단어의 origin 정보:")
sample_words = ['손', '가락', '학', '교', '물', '질', '밤', '길', '공', '간']
for word in sample_words:
    matches = df_v7[(df_v7['word'] == word) & (df_v7['pos'] == '명사')]
    if len(matches) > 0:
        row = matches.iloc[0]
        origin = row['origin']
        origin_lang = row['origin_lang']
        print(f"  {word}: origin='{origin}', origin_lang='{origin_lang}'")

Origin 필드 진단

전체 명사: 395,985개
origin 필드 non-null: 349,107개
origin 필드 null/empty: 46,878개

origin 필드 값 분포 (상위 20개):
origin
flash             21
center            15
dicyanodiamide    14
pitch             13
DC                13
軸                 12
gesture           12
令                 12
SA                12
block             12
cut               12
行                 12
stroke            12
platform          11
bridge            11
swing             11
再生                11
ATS               10
線                 10
PM                10
Name: count, dtype: int64

origin_lang 필드 값 분포 (상위 20개):
origin_lang
한자           261653
영어            32634
고유어+한자        30788
영어+한자          7782
안 밝힘           2838
안 밝힘+한자        2622
/(병기)+한자       2358
프랑스어           1278
영어+고유어         1051
일본어             942
독일어             628
이탈리아어           544
라틴어             395
에스파냐어           303
그리스어            263
독일어+한자          230
영어+안 밝힘         225
고유어+안 밝힘        200
영어+고유어+한자       164
러시아어     

---

## 3️⃣ 로마자 기반 분석 함수

In [7]:
# 음운 분석 함수: utils_phonology.py에서 로드됨
# get_final_sound_type, is_plain_obstruent (격음 제외 수정 완료)
print("음운 분석 함수: utils_phonology.py에서 로드 완료")
print(f"  is_plain_obstruent('Kha') = {is_plain_obstruent('Kha')} (expected: False)")
print(f"  is_plain_obstruent('GA') = {is_plain_obstruent('GA')} (expected: True)")

음운 분석 함수: utils_phonology.py에서 로드 완료
  is_plain_obstruent('Kha') = (False, None) (expected: False)
  is_plain_obstruent('GA') = (True, 'g') (expected: True)


In [8]:
# 3.2 형태소 파싱: utils_phonology.py에서 로드됨
# parse_dict_morph, parse_dict_morph_with_boundaries, parse_word_roman_by_morphemes
print("형태소 파싱 함수: utils_phonology.py에서 로드 완료")

형태소 파싱 함수: utils_phonology.py에서 로드 완료


---

## 4️⃣ 경음화 환경 검색 함수

In [ ]:
# 경음화 환경 검색 함수: utils_phonology.py에서 로드됨
# check_fortis_environment(morphemes_kor, morphemes_roman)
# detect_fortis_from_pron(morph2_roman, pron_roman)
# is_plain_obstruent(roman_str)
# get_final_sound_type(roman_str)
print("경음화 검색 함수: utils_phonology.py에서 로드 완료")

# 테스트
test_result = check_fortis_environment(['손', '가락'], ['SOn', 'GA-RAk'])
if test_result:
    print(f"  테스트 OK: 손+가락 → {test_result[0]['morph1_final_type']}+{test_result[0]['morph2_initial']}")
else:
    print("  테스트: 손+가락 환경 아님 (정상 — 손 끝이 n, 가락 시작이 G)")

In [10]:
# 5.1 합성어 유형 분류: utils_phonology.py에서 로드됨
# classify_compound_type (N+N 형식 사용)
print("classify_compound_type: utils_phonology.py에서 로드 완료")
print(f"  classify_compound_type('고유어', '고유어') = {classify_compound_type('고유어', '고유어')}")
print(f"  classify_compound_type('한자어', '고유어') = {classify_compound_type('한자어', '고유어')}")

classify_compound_type: utils_phonology.py에서 로드 완료
  classify_compound_type('고유어', '고유어') = N+N
  classify_compound_type('한자어', '고유어') = S+N


In [11]:
# classify_compound_type 충돌 해결
# 이전: cell 15(N+N) vs cell 16(A/B/C) 충돌 → utils_phonology.py의 N+N 형식 사용
# A/B/C 분류가 필요하면 compound_type 값에서 파생 가능:
#   A = N+N (고유어+고유어)
#   B = S+N or L+N (한자어/외래어+고유어)
#   C = *+S (한자어 포함)
print("classify_compound_type: utils_phonology.py에서 N+N 형식 사용")

classify_compound_type: utils_phonology.py에서 N+N 형식 사용


---

## 5️⃣ 형태론적 분류 (어종)

In [12]:
# 6.1 검색 함수
def search_fortis_candidates(df, sense_origin_dict):
    """
    합성어 경음화 환경 검색 + 경계 유형 정보
    """
    candidates = []

    # 명사만 필터링
    df_noun = df[df['pos'] == '명사'].copy()
    print(f"명사: {len(df_noun):,}개")

    # dict_morph가 있고 형태소 경계가 있는 것만
    df_noun = df_noun[df_noun['dict_morph'].notna()]
    df_noun = df_noun[df_noun['dict_morph'].str.contains(r'[-+]', na=False)]
    print(f"dict_morph에 형태소 경계가 있는 명사: {len(df_noun):,}개")

    # word_roman이 있는 것만
    df_noun = df_noun[df_noun['word_roman'].notna()]
    print(f"word_roman도 있는 명사: {len(df_noun):,}개")

    for idx, row in df_noun.iterrows():
        word = row['word']
        dict_morph = row['dict_morph']
        word_roman = row['word_roman']

        # 경계 유형 정보 포함
        morphemes_kor, boundaries = parse_dict_morph_with_boundaries(dict_morph)
        morphemes_roman = parse_word_roman_by_morphemes(word_roman, morphemes_kor)

        if len(morphemes_kor) < 2:
            continue

        env_list = check_fortis_environment(morphemes_kor, morphemes_roman)

        if env_list:
            for env in env_list:
                # 해당 위치의 경계 유형 가져오기
                morph_boundary_type = boundaries[env['position']] if env['position'] < len(boundaries) else ''

                # 발음 정보
                pron = str(row.get('pron', '')) if pd.notna(row.get('pron', '')) else ''
                pron_roman = str(row.get('pron_roman', '')) if pd.notna(row.get('pron_roman', '')) else ''

                # 경음화 자동 감지
                detected_fortis, fortis_type = detect_fortis_from_pron(env['morph2_roman'], pron_roman)

                # 어종 분류: 튜플 언패킹 (compound_type, morph1_origin, morph2_origin)
                compound_type, morph1_origin, morph2_origin = classify_compound_type_from_row(row, sense_origin_dict)

                # 새 컬럼
                m2_initial_place = get_m2_initial_place(env['morph2_roman'])
                m2_has_laryngeal = has_laryngeal_in_morph(env['morph2_roman'])
                word_length = len(word_roman.split('-')) if word_roman else 0

                candidate = {
                    # 기본
                    'word': word,
                    'word_stem': row['word_stem'],
                    'sense_id': row.get('sense_id', ''),

                    # 환경
                    'morph1_final_type': env['morph1_final_type'],
                    'morph2_initial': env['morph2_initial'],

                    # 경음화
                    'detected_fortis': detected_fortis,
                    'fortis_type': fortis_type if fortis_type else '',

                    # 형태론
                    'morph1_origin': morph1_origin,
                    'morph2_origin': morph2_origin,
                    'compound_type': compound_type,

                    # 경계 유형 (합성/파생)
                    'morph_boundary_type': morph_boundary_type,

                    # 새 컬럼
                    'm2_initial_place': m2_initial_place,
                    'm2_has_laryngeal': m2_has_laryngeal,
                    'word_length': word_length,

                    # 형태소
                    'dict_morph': dict_morph,
                    'word_roman': word_roman,
                    'morph1': env['morph1'],
                    'morph2': env['morph2'],
                    'morph1_roman': env['morph1_roman'],
                    'morph2_roman': env['morph2_roman'],
                    'position': env['position'],

                    # 비교용
                    'seg_morph': row.get('seg_morph', ''),
                    'anal_morph': row.get('anal_morph', ''),
                    'origin': row.get('origin', ''),

                    # 발음
                    'pron': pron,
                    'pron_roman': pron_roman,

                    # 빈도
                    'freq_LS_total': row.get('freq_LS_total', 0),
                    'freq_MP_total': row.get('freq_MP_total', 0),
                    'freq_06b': row.get('freq_06b', 0),
                    'freq_13a': row.get('freq_13a', 0),

                    # 뜻풀이
                    'definition': row.get('definition', ''),
                }

                candidates.append(candidate)

    return pd.DataFrame(candidates)

print("검색 함수 준비 완료")

검색 함수 준비 완료


In [13]:
# 6.2 검색 실행
print("합성어 경음화 환경 검색 중...\n")
df_candidates = search_fortis_candidates(df_v7, sense_origin_dict)

print(f"\n✅ 검색 완료: {len(df_candidates):,}개 후보")
print(f"\n【형태소 끝소리 유형별】")
print(df_candidates['morph1_final_type'].value_counts())
print(f"\n【합성어 유형별】")
print(df_candidates['compound_type'].value_counts())
print(f"\n【어종 분포】")
print(f"  morph1_origin:")
print(df_candidates['morph1_origin'].value_counts())
print(f"\n  morph2_origin:")
print(df_candidates['morph2_origin'].value_counts())

합성어 경음화 환경 검색 중...

명사: 395,985개
dict_morph에 형태소 경계가 있는 명사: 214,557개
word_roman도 있는 명사: 214,557개

✅ 검색 완료: 128,164개 후보

【형태소 끝소리 유형별】
morph1_final_type
vowel        47101
nasal        46325
obstruent    24149
liquid       10589
Name: count, dtype: int64

【합성어 유형별】
compound_type
S+S    81661
U+U    24332
N+N    20779
L+L     1392
Name: count, dtype: int64

【어종 분포】
  morph1_origin:
morph1_origin
한자어        81661
unknown    24332
고유어        20779
외래어         1392
Name: count, dtype: int64

  morph2_origin:
morph2_origin
한자어        81661
unknown    24332
고유어        20779
외래어         1392
Name: count, dtype: int64


In [14]:
# 6.3 결과 미리보기
print("\n상위 20개 (빈도순):\n")
df_candidates[['word', 'dict_morph', 'compound_type', 'morph1_origin', 'morph2_origin', 'detected_fortis', 'pron', 'freq_LS_total']].sort_values('freq_LS_total', ascending=False).head(20)


상위 20개 (빈도순):



,word,dict_morph,compound_type,morph1_origin,morph2_origin,detected_fortis,pron,freq_LS_total
113764,이번,이-번,U+U,unknown,unknown,no,이번,4333
108447,요즘,요-즘,N+N,고유어,고유어,no,요즘,2796
5817,관계자,관계-자,S+S,한자어,한자어,no,관계자,2358
69077,가능성,가능-성,S+S,한자어,한자어,yes,가ː능썽,1815
12350,청와대,청와-대,S+S,한자어,한자어,no,청와대,1497
110675,위원장,위원-장,S+S,한자어,한자어,no,위원장,1377
85310,새누리당,새누리-당,U+U,unknown,unknown,no,새누리당,1249
121586,전문가,전문-가,S+S,한자어,한자어,no,전문가,1155
7238,지난달,지난-달,N+N,고유어,고유어,no,지난달,1143
35835,그동안,그-동안,N+N,고유어,고유어,no,그동안,849


In [15]:
# 7.1 CSV 저장
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_path = f'{RESULT_DIR}/fortis_compound_candidates_{timestamp}.csv'

# 검토용 빈 컬럼 추가
df_candidates['review_status'] = 'pending'
df_candidates['notes'] = ''

# 컬럼 순서 정리
column_order = [
    # 기본
    'word', 'word_stem', 'sense_id',

    # 환경
    'morph1_final_type', 'morph2_initial',

    # 경음화
    'detected_fortis', 'fortis_type',

    # 형태론
    'morph1_origin', 'morph2_origin', 'compound_type',

    # ⭐ 경계 유형 (합성/파생)
    'morph_boundary_type',

    # ⭐ 새 컬럼
    'm2_initial_place', 'm2_has_laryngeal', 'word_length',

    # 형태소
    'dict_morph', 'word_roman',
    'morph1', 'morph2', 'morph1_roman', 'morph2_roman', 'position',
    'seg_morph', 'anal_morph', 'origin',

    # 발음
    'pron', 'pron_roman',

    # 빈도
    'freq_LS_total', 'freq_MP_total', 'freq_06b', 'freq_13a',

    # 뜻풀이
    'definition',

    # 검토용
    'review_status', 'notes'
]

df_full = df_candidates[column_order].sort_values(['compound_type', 'freq_LS_total'], ascending=[True, False])
df_full.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"✅ 결과 저장: {output_path}")
print(f"   총 {len(df_full):,}개 후보")
print(f"\n📋 개선 사항:")
print(f"   ✅ utils_phonology.py 공유 모듈 사용")
print(f"   ✅ sense_no 기반 어종 분류 (build_sense_origin_dict)")
print(f"   ✅ is_plain_obstruent 격음 제외 버그 수정")
print(f"   ✅ classify_compound_type 충돌 해결 (N+N 형식)")
print(f"   ✅ detect_fortis_from_pron 정의 누락 해결")
print(f"   ⭐ m2_initial_place, m2_has_laryngeal, word_length 추가")

✅ 결과 저장: /content/drive/MyDrive/DATA_2026/30_search_dictionary/search_results/fortis_compound_candidates_20260312_071626.csv
   총 128,164개 후보

📋 개선 사항:
   ✅ utils_phonology.py 공유 모듈 사용
   ✅ sense_no 기반 어종 분류 (build_sense_origin_dict)
   ✅ is_plain_obstruent 격음 제외 버그 수정
   ✅ classify_compound_type 충돌 해결 (N+N 형식)
   ✅ detect_fortis_from_pron 정의 누락 해결
   ⭐ m2_initial_place, m2_has_laryngeal, word_length 추가


In [16]:
# 7.2 통계 요약
print("\n" + "="*70)
print("📊 합성어 경음화 환경 검색 요약")
print("="*70)

print(f"\n총 후보 수: {len(df_full):,}개")

print(f"\n【형태소 끝소리 유형】")
print(df_full['morph1_final_type'].value_counts())

print(f"\n【합성어 유형】")
print(df_full['compound_type'].value_counts())

print(f"\n【경음화 자동 감지】")
print(df_full['detected_fortis'].value_counts())

print(f"\n【합성어 유형 × 경음화 교차표】")
print(pd.crosstab(df_full['compound_type'], df_full['detected_fortis']))

print(f"\n【Freq_2009 커버리지】")
print(f"  freq_06b > 0: {(df_full['freq_06b'] > 0).sum():,}개")
print(f"  freq_13a > 0: {(df_full['freq_13a'] > 0).sum():,}개")

print("\n" + "="*70)


📊 합성어 경음화 환경 검색 요약

총 후보 수: 128,164개

【형태소 끝소리 유형】
morph1_final_type
vowel        47101
nasal        46325
obstruent    24149
liquid       10589
Name: count, dtype: int64

【합성어 유형】
compound_type
S+S    81661
U+U    24332
N+N    20779
L+L     1392
Name: count, dtype: int64

【경음화 자동 감지】
detected_fortis
no         77955
yes        39515
unknown    10694
Name: count, dtype: int64

【합성어 유형 × 경음화 교차표】
detected_fortis     no  unknown    yes
compound_type                         
L+L                  1     1391      0
N+N              11181     1514   8084
S+S              57113      154  24394
U+U               9660     7635   7037

【Freq_2009 커버리지】
  freq_06b > 0: 25,038개
  freq_13a > 0: 29,329개

